# 노트북 역할 설명
## 해당 노트북은 법인별 수작업 데이터를 하나의 공통된 표준 데이터마트로 합산하기 위한 파일입니다.
### 법인별로 각자 다른 포맷의 엑셀파일을 활용하고 있으므로, 법인별 수작업 데이터의 형태를 파악하기 위해서는 'manual_data_by_entity.md' 파일을 참고하세요.


### ✅ '의뢰' 도메인에 대한 수작업 데이터 표준 데이터마트 생성
#### '의뢰' 표준 데이터 마트 컬럼 구성
- req_type_nm : 의뢰 접수 타입
- dev_comp_id : 개발법인 코드
- req_nation_cd : 의뢰법인 코드
- request_date : 의뢰 날짜
- gcc_category : GCC(Global Control Center)에서 관리하는 카테고리 대분류 기준
- proc_status : 의뢰 상태
- product_code : 신제품접수코드
- bulk_cod : 벌크구분코드
- mitem : 벌크코드
- item2: 완제품코드
- forml_code: 소제형코드
- product_name: 접수 제품명(신제품접수코드 이름)
- bulk_name: 접수 벌크명 (벌크구분코드 이름)
- customer_name : 고객사명
- brand_name : 브랜드명
- last_labno: 최종 랩넘버
- requester_id: 의뢰법인 담당연구원 사번
- requester_name: 의뢰법인 담당연구원명
- req_teamnm: 의뢰법인 담당연구원팀명
- developer_id: 개발법인 담당연구원 사번
- developer_name: 개발법인 담당연구원명
- dev_teamnm: 개발법인 담당연구원팀명
- reg_id: 의뢰법인의 등록자 id
- reg_nm: 의뢰법인의 등록자 명
- reg_deptid: 의뢰법인의 등록자 팀id
- reg_team: 의뢰법인 등록자 팀
- req_id: 의뢰법인의 마케팅 담당자 id
- req_nm: 의뢰법인의 마케팅 담당자 명
- req_team: 의뢰법인의 마케팅 팀
- req_num: 의뢰번호
- del_yn: 삭제여부
- memo: 비고

#### 표준 데이터 마트 생성 순서
1. 모든 법인에 대한 수작업 '의뢰'데이터를 로드한다. (각 법인 엑셀 데이터 중 의뢰 데이터 원본 시트만 불러오기)
2. 표준 데이터마트 컬럼(시스템DB 컬럼과 동일한 컬럼명을 사용) 형태로 모든 법인의 수작업 의뢰 데이터를 구성한다. (참고 'column_mapping_NPD.md' )


#### 표준 데이터 마트 생성 규칙
1. 표준 데이터마트 컬럼을 기준으로 수작업 '의뢰' 데이터를 재구성하되, 없는 정보는 빈 값으로 비워두면 됨
2. 신제품접수코드와 벌크구분코드는 추후 '시스템 DB'와 매핑할 가장 중요한 컬럼임. 띄어쓰기, 쉼표, 따옴표, 대쉬 등을 삭제하는 기본적인 전처리가 필요함.

---
## Step 1. 법인별 수작업 데이터 로드
> 각 법인의 엑셀 파일에서 의뢰(NPD) Raw Data 시트만 불러옵니다.
> 법인별 파일명 및 시트명은 `manual_data_by_entity.md` 참고

| 법인 | comp_id | 파일명 | Raw Data 시트명 |
|---|---|---|---|
| 상해 | 5100 | 상해_NPD.xlsx | `Raw Data` |
| 광저우 | 5200 | 광저우_수작업.xlsx | `전체` |
| 미국 | 6400 | 미국_수작업.xlsx | `US NPD request` |
| 태국 | 7200 | 태국_수작업.xlsx | `NPD 로우 데이터` |
| 인도네시아 | 7100 | 인니_수작업.xlsx | `Raw NPD Request 2026` |

In [73]:
import pandas as pd
import os

# 수작업 엑셀 파일이 저장된 기본 경로
DATA_DIR = "./manual_data"

In [74]:
# ─────────────────────────────────────────────
# 법인별 파일명 / NPD Raw Data 시트명 / 컬럼 헤더 행 정의
# header : 엑셀에서 실제 컬럼명이 위치한 행 (0-indexed)
# ─────────────────────────────────────────────
ENTITY_CONFIG = {
    "상해":   {"comp_id": "5100", "file": "상해_NPD.xlsx",      "sheet": "Raw Data",             "header": 1},
    "광저우": {"comp_id": "5200", "file": "광저우_수작업.xlsx",  "sheet": "전체",                  "header": 0},
    "미국":   {"comp_id": "6400", "file": "미국_수작업.xlsx",    "sheet": "US NPD request",       "header": 8},
    "태국":   {"comp_id": "7200", "file": "태국_수작업.xlsx",    "sheet": "NPD 로우 데이터",       "header": 2},
    "인니":   {"comp_id": "7100", "file": "인니_수작업.xlsx",    "sheet": "Raw NPD Request 2026", "header": 0},
}

# ─────────────────────────────────────────────
# 법인별 엑셀 로드 (의뢰 Raw Data 시트만 읽기)
# ─────────────────────────────────────────────
import warnings

raw_data = {}  # { 법인명: DataFrame }

for entity, cfg in ENTITY_CONFIG.items():
    file_path = os.path.join(DATA_DIR, cfg["file"])

    # openpyxl의 Data Validation 미지원 경고는 데이터에 영향 없으므로 무시
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        df = pd.read_excel(file_path, sheet_name=cfg["sheet"], header=cfg["header"], dtype=str)

    # 로드 결과 간단 확인 (행 수, 열 수, 첫 번째 컬럼명)
    print(f"[{entity}] comp_id={cfg['comp_id']} | shape={df.shape} | 첫 컬럼='{df.columns[0]}'")

    raw_data[entity] = df

[상해] comp_id=5100 | shape=(23210, 38) | 첫 컬럼='Unnamed: 0'
[광저우] comp_id=5200 | shape=(52188, 47) | 첫 컬럼='순번'
[미국] comp_id=6400 | shape=(3292, 21) | 첫 컬럼='Unnamed: 0'
[태국] comp_id=7200 | shape=(5888, 14) | 첫 컬럼='연도'
[인니] comp_id=7100 | shape=(2302, 13) | 첫 컬럼='Request date'


---
## Step 2. 표준 데이터마트 생성

### 2-1. 법인별 컬럼 매핑 딕셔너리 정의
> `column_mapping_NPD.md` 기준으로 수작업 컬럼명 → 표준 컬럼명 매핑
> `-` (매핑 없음) 컬럼은 딕셔너리에서 제외 → 변환 후 NaN으로 처리
> 혼합 사용 컬럼(태국 `NPD Code`, 상해/태국 `제품명`/`Product Name`)은 2-3에서 별도 처리

In [84]:
# ─────────────────────────────────────────────────────────────────────────────
# 법인별 수작업 컬럼명 → 표준 데이터마트 컬럼명 매핑 딕셔너리
# 기준 문서 : column_mapping_NPD.md
#
# 작성 규칙
#   - key   : 수작업 엑셀의 원본 컬럼명 (정규화 후 기준 — \n 제거, 앞뒤 공백 제거)
#   - value : 표준 데이터마트 컬럼명 (시스템DB 컬럼명과 동일)
#   - 매핑 없는 컬럼(-) 은 딕셔너리에서 제외 → 변환 후 NaN으로 자동 처리
#   - 복수 컬럼을 합산해야 하는 경우는 CONCAT_COLUMNS 에 별도 정의
#   - 혼합 사용 컬럼(1개 수작업 컬럼 → 2개 표준 컬럼)은 ※ 주석으로 표시,
#     실제 분리 로직은 Step 2-3 예외 처리에서 수행
# ─────────────────────────────────────────────────────────────────────────────

COLUMN_MAPPING = {

    # ── 상해 (comp_id: 5100) ──────────────────────────────────────────────
    "상해": {
        "구분":                   "req_type_nm",
        "의뢰일자":               "request_date",
        "카테고리":               "gcc_category",
        "상태":                   "proc_status",
        "신제품접수코드":         "product_code",
        "벌크구분코드":           "bulk_cod",
        "SAP 벌크코드":           "mitem",
        "SAP 제품코드":           "item2",
        "제형코드":               "forml_code",
        # ※ "제품명" → product_name 과 bulk_name 에 혼용 (2-3에서 처리)
        "고객사명":               "customer_name",
        "브랜드명":               "brand_name",
        "(의뢰법인)제형 연구원":  "requester_name",   # 정규화 후: \n 제거
        "(의뢰법인)제형팀":       "req_teamnm",       # 정규화 후: \n 제거
        "(개발법인)제형연구원":   "developer_name",   # 정규화 후: \n 제거
        "(개발법인)제형팀":       "dev_teamnm",       # 정규화 후: \n 제거
        "마케팅담당자":           "req_nm",
        "마케팅팀":               "req_team",
        "삭제여부":               "del_yn",
        "비고(是否储备品)":       "memo",             # 실제 컬럼명 반영
    },

    # ── 광저우 (comp_id: 5200) ────────────────────────────────────────────
    "광저우": {
        "구분":                     "req_type_nm",
        "의뢰일자":                 "request_date",
        "扩大":                     "gcc_category",
        "상태":                     "proc_status",
        "신제품접수코드":           "product_code",
        "벌크구분코드":             "bulk_cod",
        "SAP 벌크코드":             "mitem",
        "SAP 제품코드":             "item2",
        "제형코드":                 "forml_code",
        "제품명":                   "product_name",
        "벌크명":                   "bulk_name",
        "고객사명":                 "customer_name",
        "브랜드명":                 "brand_name",
        "랩넘버":                   "last_labno",
        "(의뢰법인)제형연구원":     "requester_name",
        "(의뢰법인)제형팀":         "req_teamnm",
        "(개발법인)제형연구원":     "developer_name",
        "(개발법인)제형팀":         "dev_teamnm",
        "마케팅담당자":             "req_nm",
        "마케팅팀":                 "req_team",
        "삭제여부":                 "del_yn",
        "Sample/Benchmark/Lab.#":   "memo",
    },

    # ── 미국 (comp_id: 6400) ──────────────────────────────────────────────
    # memo → CONCAT_COLUMNS 에서 Entity | Status | OTC(Sunscreen) 합산 처리
    "미국": {
        "Request Date":      "request_date",
        "Category(2308)":    "gcc_category",
        "NPD Product No":    "product_code",
        "NPD Bulk No":       "bulk_cod",
        "Small class code":  "forml_code",
        "Product Name":      "product_name",
        "Bulk Name":         "bulk_name",
        "Customer":          "customer_name",
        "Brand":             "brand_name",
        "Chemist":           "developer_name",
        "Team":              "dev_teamnm",
        "Sales name":        "req_nm",
    },

    # ── 태국 (comp_id: 7200) ──────────────────────────────────────────────
    "태국": {
        "NPS Date":              "request_date",
        "회의용 분류":            "gcc_category",
        # ※ "NPD Code" → product_code(P9%) 와 bulk_cod(P3%) 혼용 (2-3에서 Prefix로 분리)
        # ※ "Product Name" → product_name 과 bulk_name 에 혼용 (2-3에서 처리)
        "Customer Name(Will be updated automatically When you add correct NPD code)":         "customer_name",
        "Brand":                 "brand_name",
        "Person in charge (MK)": "req_nm",
        "Sub Categofy":          "memo",
    },

    # ── 인도네시아 (comp_id: 7100) ────────────────────────────────────────
    # memo → CONCAT_COLUMNS 에서 Formulation | New/existing 합산 처리
    "인니": {
        "Classification":    "req_type_nm",
        "Request date":      "request_date",
        "Extended Category": "gcc_category",
        "NPD Product No":    "product_code",
        "NPD Bulk No":       "bulk_cod",
        "Bulk name":         "bulk_name",
        "Customer":          "customer_name",
        "Brand":             "brand_name",
        "Sales In Charge":   "req_nm",
        "Sales team":        "req_team",
    },
}

# ─────────────────────────────────────────────────────────────────────────────
# 복수 컬럼을 하나의 표준 컬럼에 | 구분자로 합산하는 매핑 정의
# 기준 문서 : column_mapping_NPD.md > 비고 컬럼 섹션
#
# 구조 : { 법인명: { 표준컬럼명: [합산할 원본 컬럼명 리스트] } }
# ─────────────────────────────────────────────────────────────────────────────
CONCAT_COLUMNS = {
    "미국": {
        "memo": ["Entity", "Status", "OTC(Sunscreen)"],
    },
    "인니": {
        "memo": ["Formulation", "New/existing"],
    },
}

print("컬럼 매핑 딕셔너리 정의 완료")
for entity, mapping in COLUMN_MAPPING.items():
    concat_info = CONCAT_COLUMNS.get(entity, {})
    print(f"  [{entity}] 단일 매핑: {len(mapping)}개 | 합산 매핑: {len(concat_info)}개")

컬럼 매핑 딕셔너리 정의 완료
  [상해] 단일 매핑: 19개 | 합산 매핑: 0개
  [광저우] 단일 매핑: 22개 | 합산 매핑: 0개
  [미국] 단일 매핑: 12개 | 합산 매핑: 1개
  [태국] 단일 매핑: 6개 | 합산 매핑: 0개
  [인니] 단일 매핑: 10개 | 합산 매핑: 1개


### 2-2. 표준 데이터마트 변환 함수 작성
> 법인별 DataFrame을 표준 컬럼 구조로 변환하는 함수 모음입니다.
> 실행 순서: `normalize_columns` → `apply_exceptions`(2-3) → `apply_concat_columns` → rename → 컬럼 정렬

In [85]:
# 표준 데이터마트 컬럼 순서 정의 — 시스템DB 컬럼명과 동일한 순서로 고정
STANDARD_COLUMNS = [
    "req_type_nm", "dev_comp_id", "req_nation_cd", "request_date",
    "gcc_category", "proc_status", "product_code", "bulk_cod",
    "mitem", "item2", "forml_code", "product_name", "bulk_name",
    "customer_name", "brand_name", "last_labno",
    "requester_id", "requester_name", "req_teamnm",
    "developer_id", "developer_name", "dev_teamnm",
    "reg_id", "reg_nm", "reg_deptid", "reg_team",
    "req_id", "req_nm", "req_team", "req_num", "del_yn", "memo",
]


def normalize_str(s: str) -> str:
    # \n 및 모든 공백(앞뒤·중간) 제거 — df 컬럼과 매핑 key 양쪽에 동일하게 적용해 매칭 보장
    return "".join(s.split())


def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    # df 컬럼명 전체에 normalize_str 적용
    df.columns = [normalize_str(c) for c in df.columns]
    return df


def apply_concat_columns(df: pd.DataFrame, entity: str) -> pd.DataFrame:
    # CONCAT_COLUMNS 정의 기준으로 복수 원본 컬럼을 | 구분자로 합산해 표준 컬럼에 저장
    concat_map = CONCAT_COLUMNS.get(entity, {})
    for target_col, source_cols in concat_map.items():
        normalized_sources = [normalize_str(c) for c in source_cols]
        existing = [c for c in normalized_sources if c in df.columns]
        missing  = [c for c in normalized_sources if c not in df.columns]
        if missing:
            print(f"  ⚠ [{entity}] 합산 대상 컬럼 없음: {missing}")
        if existing:
            df[target_col] = (
                df[existing]
                .fillna("")
                .apply(lambda row: " | ".join(v for v in row if v != ""), axis=1)
                .replace("", pd.NA)
            )
    return df


print("변환 함수 정의 완료 : normalize_str / normalize_columns / apply_concat_columns")

변환 함수 정의 완료 : normalize_str / normalize_columns / apply_concat_columns


### 2-3. 혼합 사용 컬럼 예외 처리
> 1개 원본 컬럼이 2개 표준 컬럼에 매핑되는 케이스를 처리합니다.
> `normalize_columns` 적용 후 정규화된 컬럼명 기준으로 동작합니다.

| 법인 | 원본 컬럼 | 처리 방식 |
|---|---|---|
| 상해 | `제품명` | `product_name`, `bulk_name` 양쪽에 동일 값 복사 |
| 태국 | `NPD Code` | Prefix `P9%` → `product_code` / `P3%` → `bulk_cod` 분리 |
| 태국 | `Product Name` | `product_name`, `bulk_name` 양쪽에 동일 값 복사 |

In [86]:
def apply_exceptions(df: pd.DataFrame, entity: str) -> pd.DataFrame:
    # 혼합 사용 컬럼(1개 원본 → 2개 표준 컬럼) 처리 — normalize_columns 적용 후 호출
    if entity == "상해":
        # 제품명 컬럼이 product_name / bulk_name 혼용 → 양쪽에 동일 값 복사
        if "제품명" in df.columns:
            df["product_name"] = df["제품명"]
            df["bulk_name"]    = df["제품명"]

    elif entity == "태국":
        # NPDCode: P9 prefix → product_code, P3 prefix → bulk_cod 로 분리
        if "NPDCode" in df.columns:
            df["product_code"] = df["NPDCode"].where(df["NPDCode"].str.startswith("P9", na=False))
            df["bulk_cod"]     = df["NPDCode"].where(df["NPDCode"].str.startswith("P3", na=False))

        # ProductName 컬럼이 product_name / bulk_name 혼용 → 양쪽에 동일 값 복사
        if "ProductName" in df.columns:
            df["product_name"] = df["ProductName"]
            df["bulk_name"]    = df["ProductName"]

    return df


def to_standard_datamart(df: pd.DataFrame, entity: str) -> pd.DataFrame:
    # 법인별 수작업 DataFrame → 표준 데이터마트 형태로 변환하는 메인 함수
    mapping = COLUMN_MAPPING[entity]

    # 1) 수작업 df 컬럼명 정규화 (\n·공백 제거)
    df = normalize_columns(df.copy())

    # 2) 혼합 사용 컬럼 예외 처리 (1개 원본 → 2개 표준 컬럼 분리)
    df = apply_exceptions(df, entity)

    # 3) 매핑 key도 동일 기준으로 정규화해 df 컬럼명과 비교
    normalized_mapping = {normalize_str(k): v for k, v in mapping.items()}
    actual_cols = set(df.columns)
    missing_keys = [k for k in normalized_mapping if k not in actual_cols]
    if missing_keys:
        print(f"  ⚠ [{entity}] 매핑 key 불일치 (파일에 없는 컬럼): {missing_keys}")
    else:
        print(f"  ✓ [{entity}] 매핑 key 전체 일치")

    # 4) 복수 컬럼 합산 처리 (CONCAT_COLUMNS 기준, | 구분자)
    df = apply_concat_columns(df, entity)

    # 5) 컬럼명 rename → 매핑 없는 컬럼은 제외
    valid_mapping = {k: v for k, v in normalized_mapping.items() if k in actual_cols}
    df_renamed = df.rename(columns=valid_mapping)

    # 6) 표준 컬럼 중 없는 항목은 NaN으로 채워 컬럼 순서 통일
    for col in STANDARD_COLUMNS:
        if col not in df_renamed.columns:
            df_renamed[col] = pd.NA

    df_standard = df_renamed[STANDARD_COLUMNS].copy()
    return df_standard


print("변환 함수 정의 완료 : apply_exceptions / to_standard_datamart")

변환 함수 정의 완료 : apply_exceptions / to_standard_datamart


---
## Step 3. 법인별 표준 데이터마트 변환 실행 및 정제

### 3-1. 변환 실행
> `to_standard_datamart` 함수로 법인별 Raw DataFrame을 표준 컬럼 구조로 변환합니다.

### 3-2. 데이터 정제
1. **전체 NaN 행 제거** : 모든 표준 컬럼이 빈 값인 행은 의미 없는 데이터이므로 제거합니다.
2. **2026년 이후 데이터만 유지** : `request_date` 기준 2026-01-01 이후 데이터만 분석 대상으로 남깁니다.

In [87]:
DATE_CUTOFF = pd.Timestamp("2026-01-01")  # 분석 대상 기준일 — 이 날짜 이후 데이터만 유지

standard_data = {}  # { 법인명: 정제 완료된 표준 DataFrame }

for entity, cfg in ENTITY_CONFIG.items():
    # 3-1. 표준 데이터마트 형태로 변환
    df = to_standard_datamart(raw_data[entity], entity)
    before_all = len(df)

    # 3-2-1. 전체 NaN 행 제거 — 모든 표준 컬럼이 비어 있는 행은 유효 데이터가 아님
    df = df.dropna(how="all")
    after_dropna = len(df)

    # 3-2-2. request_date 파싱 — 날짜 비교를 위해 datetime 타입으로 변환 (파싱 불가 값은 NaT)
    df["request_date"] = pd.to_datetime(df["request_date"], errors="coerce")

    # 3-2-3. 2026-01-01 이후 데이터만 유지 — request_date가 NaT인 행은 제외
    df = df[df["request_date"] >= DATE_CUTOFF]
    after_date = len(df)

    standard_data[entity] = df

    print(
        f"[{entity}] 변환 전: {before_all}행 "
        f"→ NaN행 제거 후: {after_dropna}행 "
        f"→ 2026년 이후: {after_date}행"
    )

  ✓ [상해] 매핑 key 전체 일치
[상해] 변환 전: 23210행 → NaN행 제거 후: 23210행 → 2026년 이후: 3434행
  ✓ [광저우] 매핑 key 전체 일치
[광저우] 변환 전: 52188행 → NaN행 제거 후: 52180행 → 2026년 이후: 6175행
  ✓ [미국] 매핑 key 전체 일치
[미국] 변환 전: 3292행 → NaN행 제거 후: 3292행 → 2026년 이후: 386행
  ✓ [태국] 매핑 key 전체 일치
[태국] 변환 전: 5888행 → NaN행 제거 후: 5888행 → 2026년 이후: 888행
  ✓ [인니] 매핑 key 전체 일치
[인니] 변환 전: 2302행 → NaN행 제거 후: 2301행 → 2026년 이후: 1095행


In [93]:
# product_code와 bulk_cod 둘 다 없는 행 — 시스템 DB 매핑이 불가한 미식별 의뢰 건 파악용
for entity, df in standard_data.items():
    no_code = df[df["product_code"].isna() & df["bulk_cod"].isna()]
    total   = len(df)
    pct     = len(no_code) / total * 100 if total else 0
    print(f"[{entity}] product_code & bulk_cod 모두 없는 행: {len(no_code)}건 / 전체 {total}행 ({pct:.1f}%)")

    # 월별 집계 — request_date 기준 그룹화, 법인 전체 월별 건수 대비 비율도 함께 출력
    monthly_no_code  = no_code["request_date"].dt.to_period("M").value_counts().sort_index()
    monthly_total    = df["request_date"].dt.to_period("M").value_counts()  # 법인 전체 월별 건수
    for period, count in monthly_no_code.items():
        month_total  = monthly_total.get(period, 0)
        month_pct    = count / month_total * 100 if month_total else 0
        print(f"    {period}: {count}건 / 월 전체 {month_total}건 ({month_pct:.1f}%)")
    print()

[상해] product_code & bulk_cod 모두 없는 행: 0건 / 전체 3434행 (0.0%)

[광저우] product_code & bulk_cod 모두 없는 행: 0건 / 전체 6175행 (0.0%)

[미국] product_code & bulk_cod 모두 없는 행: 61건 / 전체 386행 (15.8%)
    2026-01: 19건 / 월 전체 118건 (16.1%)
    2026-02: 13건 / 월 전체 48건 (27.1%)
    2026-03: 29건 / 월 전체 70건 (41.4%)

[태국] product_code & bulk_cod 모두 없는 행: 17건 / 전체 888행 (1.9%)
    2026-01: 4건 / 월 전체 174건 (2.3%)
    2026-03: 4건 / 월 전체 374건 (1.1%)
    2026-04: 3건 / 월 전체 144건 (2.1%)
    2026-05: 6건 / 월 전체 151건 (4.0%)

[인니] product_code & bulk_cod 모두 없는 행: 0건 / 전체 1095행 (0.0%)



In [94]:
OUTPUT_DIR = "./manual_to_standard_data"  # 법인별 표준 데이터마트 CSV 저장 경로
os.makedirs(OUTPUT_DIR, exist_ok=True)   # 폴더 없으면 생성, 있으면 그대로

for entity, df in standard_data.items():
    file_path = os.path.join(OUTPUT_DIR, f"{entity}_standard.csv")
    df.to_csv(file_path, index=False, encoding="utf-8-sig")  # utf-8-sig: 엑셀에서 한글 깨짐 방지
    print(f"[{entity}] 저장 완료 → {file_path} ({len(df)}행)")

[상해] 저장 완료 → ./manual_to_standard_data/상해_standard.csv (3434행)
[광저우] 저장 완료 → ./manual_to_standard_data/광저우_standard.csv (6175행)
[미국] 저장 완료 → ./manual_to_standard_data/미국_standard.csv (386행)
[태국] 저장 완료 → ./manual_to_standard_data/태국_standard.csv (888행)
[인니] 저장 완료 → ./manual_to_standard_data/인니_standard.csv (1095행)
